# Exploratory Data Analysis - Credit Card Default

This notebook is only for **understanding the data**. The real training workflow runs from the Python scripts (`python -m src.train` or `docker compose run --rm trainer`) and does **not** depend on this notebook.

**Before running:** put the Kaggle CSV at `data/raw/default_credit_card_clients.csv` and start Jupyter from the project root (or from `notebooks/`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make the project's src/ package importable from the notebooks/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.data import detect_target_column, resolve_data_path

pd.set_option("display.max_columns", 30)

## 1. Load the dataset

In [ ]:
csv_path = resolve_data_path(PROJECT_ROOT / "data" / "raw" / "default_credit_card_clients.csv")
raw = pd.read_csv(csv_path)
target = detect_target_column(raw)
print("File:", csv_path.name)
print("Target column:", target)
raw.head()

## 2. Shape

In [ ]:
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")

## 3. Columns

In [ ]:
raw.dtypes.to_frame("dtype").T

## 4. Missing values

In [ ]:
missing = raw.isna().sum()
print("Total missing values:", int(missing.sum()))
missing[missing > 0]

## 5. Target distribution

In [ ]:
counts = raw[target].value_counts().sort_index()
share = raw[target].value_counts(normalize=True).sort_index()
display(pd.DataFrame({"customers": counts, "share": share.round(3)}).rename(index={0: "No default", 1: "Default"}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["No default", "Default"], counts.values, color=["#4C78A8", "#E45756"])
ax.set_title("Default next month")
ax.set_ylabel("Customers")
plt.show()

## 6. Descriptive statistics

In [ ]:
raw[["LIMIT_BAL", "AGE", "BILL_AMT1", "PAY_AMT1", "PAY_0"]].describe().round(1)

## 7. A few useful charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Default rate by the most recent repayment status (PAY_0).
rate_by_pay0 = raw.groupby("PAY_0")[target].mean()
axes[0].bar(rate_by_pay0.index.astype(str), rate_by_pay0.values, color="#E45756")
axes[0].set_title("Default rate by last repayment status (PAY_0)")
axes[0].set_xlabel("PAY_0 (-2/-1/0 = on time, 1+ = months late)")
axes[0].set_ylabel("Default rate")

# Credit limit by default status.
raw.boxplot(column="LIMIT_BAL", by=target, ax=axes[1], showfliers=False)
axes[1].set_title("Credit limit by default status")
axes[1].set_xlabel("Default next month (0 = no, 1 = yes)")
axes[1].set_ylabel("LIMIT_BAL")

# Age distribution by default status.
for value, label, color in [(0, "No default", "#4C78A8"), (1, "Default", "#E45756")]:
    axes[2].hist(raw.loc[raw[target] == value, "AGE"], bins=30, alpha=0.6, label=label, color=color, density=True)
axes[2].set_title("Age distribution")
axes[2].set_xlabel("Age")
axes[2].legend()

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Undocumented category codes (grouped into "others" in src/features.py).
print("EDUCATION codes:", sorted(raw["EDUCATION"].unique()))
print("MARRIAGE codes:", sorted(raw["MARRIAGE"].unique()))

## 8. Basic interpretation

Fill in after running on the real data. Things to look for:

- **Repayment status is the strongest signal:** customers who were already late last month (`PAY_0 >= 1`) should show a much higher default rate.
- **Credit limit:** compare the median `LIMIT_BAL` of defaulters and non-defaulters in the box plot.
- **Undocumented codes:** `EDUCATION` (0, 5, 6) and `MARRIAGE` (0) contain codes not described in the documentation; the pipeline groups them into "others".
- **Data quality:** check the missing-values cell above.

## 9. Class imbalance

Only a minority of customers default (check the exact share in section 5). This means:

- **Accuracy is misleading:** a model that always predicts "no default" would already get a high accuracy while catching zero defaulters.
- We use a **stratified** train/test split, `class_weight="balanced"` and metrics such as **ROC-AUC, recall, precision and F1**.

## 10. Sensitive attributes

`SEX`, `AGE`, `MARRIAGE` and `EDUCATION` are personal or demographic attributes. Using them in credit-related models can create **unfair or discriminatory outcomes** and may be restricted by regulation (e.g. EU anti-discrimination law, GDPR, the EU AI Act, which lists creditworthiness assessment as a high-risk use case).

In this prototype they are kept so we can **discuss and measure** their impact. A real deployment would require a fairness analysis (e.g. comparing recall and false-positive rates across groups) and a legal review before deciding whether these variables can be used at all.